# Figure 2: Look-Locker

The cell below is tagged `#| label: fig-look-locker-output`, so `03-look-locker.md` embeds its output with `:::{figure} #fig-look-locker-output`.

In [ ]:
#| label: fig-look-locker-output
# Figure 2: Look-Locker. Small readout pulses make the recovery look faster (T1* < T1).
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Interactive figure for the website + PNG snapshot for the PDF (same probe as the template).
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); PDF will show the caption only.")

# --- Parameters -------------------------------------------------------------
T1 = 1130.0                        # muscle at 1.5T, same tissue as Figure 1 [ms]
M0 = 1.0                           # equilibrium magnetization (arbitrary units)
TAU = 100.0                        # time between two readout pulses [ms]
N_PULSES = 50                      # 50 x 100 ms = 5 s of sampling after ONE inversion
t = np.arange(N_PULSES) * TAU      # time of each readout pulse after the inversion [ms]
t_fine = np.linspace(0, t[-1], 400)
ALPHAS = np.arange(1, 16)          # readout flip angles for the slider [deg]
DEFAULT_ALPHA = 8
TAU_EXAMPLES = [50.0, 100.0, 200.0]  # for the right panel

# --- Simulation -------------------------------------------------------------
def look_locker_samples(alpha_deg):
    """Mz just before each readout pulse, after one perfect inversion at t = 0.

    Between two pulses Mz relaxes towards M0 (T1 recovery).
    Each pulse tilts the magnetization by alpha, which keeps only cos(alpha) of Mz.
    """
    ca = np.cos(np.deg2rad(alpha_deg))
    E = np.exp(-TAU / T1)
    mz = np.empty(N_PULSES)
    mz[0] = -M0                                  # perfect inversion
    for n in range(1, N_PULSES):
        mz[n] = M0 - (M0 - mz[n - 1] * ca) * E   # pulse, then recovery for TAU
    return mz

def ll_model(time, A, B, T1_star):
    """Apparent recovery: S = A - B exp(-t / T1*)."""
    return A - B * np.exp(-time / T1_star)

def t1_star_formula(alpha_deg, tau):
    """1/T1* = 1/T1 - ln(cos alpha)/tau."""
    return 1.0 / (1.0 / T1 - np.log(np.cos(np.deg2rad(alpha_deg))) / tau)

def fit_ll(alpha_deg):
    mz = look_locker_samples(alpha_deg)
    (A, B, T1s), _ = curve_fit(ll_model, t, mz, p0=[1, 2, 1000])
    return mz, A, B, T1s

# --- Figure -----------------------------------------------------------------
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("Recovery after one inversion",
                                    "How much faster does it look?"))

# Static traces (always visible)
fig.add_trace(go.Scatter(x=t_fine, y=M0 * (1 - 2 * np.exp(-t_fine / T1)), mode="lines",
                         line=dict(color="lightgray", width=6),
                         name=f"true recovery (no readouts), T1 = {T1:.0f} ms"), row=1, col=1)
a_fine = np.linspace(1, 15, 100)
for tau in TAU_EXAMPLES:
    fig.add_trace(go.Scatter(x=a_fine, y=t1_star_formula(a_fine, tau) / T1, mode="lines",
                             name=f"pulse spacing {tau:.0f} ms"), row=1, col=2)
n_static = 1 + len(TAU_EXAMPLES)

# Per-angle traces: only one set visible at a time
TRACES_PER_ANGLE = 4
for a in ALPHAS:
    mz, A, B, T1s = fit_ll(a)
    T1_corr = T1s * (B / A - 1)          # correction that needs no flip angle
    vis = bool(a == DEFAULT_ALPHA)
    fig.add_trace(go.Scatter(x=t, y=mz, mode="markers", marker=dict(size=7, color="black"),
                             name="Mz at each readout", visible=vis), row=1, col=1)
    fig.add_trace(go.Scatter(x=t_fine, y=ll_model(t_fine, A, B, T1s), mode="lines",
                             line=dict(color="crimson", dash="dash"),
                             name=f"fit: T1* = {T1s:.0f} ms → corrected T1 = {T1_corr:.0f} ms",
                             visible=vis), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, t[-1]], y=[A, A], mode="lines",
                             line=dict(color="crimson", dash="dot", width=1),
                             name=f"level it recovers to: A = {A:.2f}", visible=vis), row=1, col=1)
    fig.add_trace(go.Scatter(x=[a], y=[T1s / T1], mode="markers",
                             marker=dict(size=14, color="crimson", symbol="diamond"),
                             name=f"current: T1*/T1 = {T1s / T1:.2f}", visible=vis), row=1, col=2)

steps = []
for i, a in enumerate(ALPHAS):
    visible = [True] * n_static + [False] * (TRACES_PER_ANGLE * len(ALPHAS))
    start = n_static + i * TRACES_PER_ANGLE
    visible[start:start + TRACES_PER_ANGLE] = [True] * TRACES_PER_ANGLE
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{a}°"))

fig.update_layout(
    sliders=[dict(active=int(np.where(ALPHAS == DEFAULT_ALPHA)[0][0]), steps=steps,
                  currentvalue=dict(prefix="Readout flip angle α: "), pad=dict(t=50))],
    height=540, legend=dict(orientation="h", y=-0.45), margin=dict(t=60),
    template="plotly_white",
)
fig.update_xaxes(title_text="Time after inversion [ms]", row=1, col=1)
fig.update_yaxes(title_text="Mz / M0", range=[-1.1, 1.1], row=1, col=1)
fig.update_xaxes(title_text="Readout flip angle α [°]", row=1, col=2)
fig.update_yaxes(title_text="T1* / T1", range=[0.5, 1.02], row=1, col=2)
fig
